In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
working_directory = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "src" / "procesado" / "limpiar_poblacion.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root containing src/procesado.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 1. Cleaning population

Read the Andalusian population Bronze table, apply the current cleaning steps, and preview the result.

In [ ]:
from src.common.storage import read_csv_dataset, s3_uri_for
from src.procesado.pipeline import clean_and_upload_population
from src.procesado.common.paths import POPULATION_BRONZE_PATH

population_bronze = read_csv_dataset(
    POPULATION_BRONZE_PATH,
    s3_uri=s3_uri_for(POPULATION_BRONZE_PATH.name, "raw/data"),
    dtype=str,
    keep_default_na=False,
)
population_cleaned = clean_and_upload_population(population_bronze)
print(
    f"Bronze rows: {len(population_bronze):,}; "
    f"cleaned rows: {len(population_cleaned):,}; "
    "outputs persisted according to SAVE_LOCALLY and SAVE_S3"
)
display(population_cleaned.head(10))

# 2. Cleaning income

Read the INE Bronze table, keep per-person income, save the Silver Parquet, and preview the result.

In [ ]:
from src.procesado.pipeline import clean_and_upload_income
from src.procesado.common.paths import INCOME_BRONZE_PATH

income_bronze = read_csv_dataset(
    INCOME_BRONZE_PATH,
    s3_uri=s3_uri_for(INCOME_BRONZE_PATH.name, "raw/data"),
    dtype=str,
    keep_default_na=False,
)
income_cleaned = clean_and_upload_income(income_bronze)
print(
    f"Bronze rows: {len(income_bronze):,}; "
    f"cleaned rows: {len(income_cleaned):,}; "
    "outputs persisted according to SAVE_LOCALLY and SAVE_S3"
)
display(income_cleaned.head(10))

# 3. Cleaning gyms

Read the Andalusian OSM Bronze gym table, clean it, save the GeoParquet Silver table, and preview the result.

In [ ]:
from src.procesado.pipeline import clean_and_upload_gyms
from src.procesado.common.paths import GYMS_BRONZE_PATH

gyms_bronze = read_csv_dataset(
    GYMS_BRONZE_PATH,
    s3_uri=s3_uri_for(GYMS_BRONZE_PATH.name, "raw/data"),
    dtype=str,
    keep_default_na=False,
)
gyms_cleaned = clean_and_upload_gyms(gyms_bronze)
print(
    f"Bronze rows: {len(gyms_bronze):,}; "
    f"Silver rows: {len(gyms_cleaned):,}; "
    f"gyms: {gyms_cleaned['sport_centre_type'].eq('gym').sum():,}; "
    f"other sports facilities: "
    f"{gyms_cleaned['sport_centre_type'].eq('other_sport_facility').sum():,}; "
    "outputs persisted according to SAVE_LOCALLY and SAVE_S3"
)
display(gyms_cleaned.head(10))

# 4. Cleaning municipality boundaries

Read the CNIG Bronze CSV, clean the boundaries, and persist the Silver GeoParquet according to the storage settings.

In [ ]:
from src.procesado.common.paths import MUNICIPALITY_BOUNDARIES_BRONZE_PATH
from src.procesado.pipeline import clean_and_upload_municipality_boundaries

cnig_bronze = read_csv_dataset(
    MUNICIPALITY_BOUNDARIES_BRONZE_PATH,
    s3_uri=s3_uri_for(MUNICIPALITY_BOUNDARIES_BRONZE_PATH.name, "raw/data"),
    dtype=str,
    keep_default_na=False,
)
cnig_cleaned = clean_and_upload_municipality_boundaries(cnig_bronze)
print(
    f"Silver municipality boundaries: {len(cnig_cleaned):,}; "
    "outputs persisted according to SAVE_LOCALLY and SAVE_S3"
)
display(cnig_cleaned.head(10))